# 📊 CEM4644 · MP6A — Tables
## Workshop (in class): *Concrete mixes and their strength*

**No coding needed.** Each grey box is one step: click ▶, wait, read the result, answer the report question. Run from top to bottom.

Photos and drawings were the last four labs. Most construction data is neither: it is a **table** (one row per mix, per building, per bid) or a **time series** (one value per hour, per day; that is MP6B). This part does the table: predict a number, predict a class, see what the model learned, then give the same job to a chat model. Every answer is checked against what really happened. About 75 minutes. No GPU needed.

In [ ]:
#@title ▶ Step 0 · Run me first (1 minute) { display-mode: "form" }
#@markdown Click ▶ and wait for the ✅ line.
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp6_tabular_timeseries", "aec_tab"
FOLDERS = ["mp6_tabular_timeseries"]          # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_tab import lab
lab.setup(dataset="workshop", part="tabular")


## Part 1 · A table

1,030 concrete mixes tested in a laboratory: what went into each cubic metre, how old the sample was, and the strength it reached. Every row is one mix; the last column, **compressive strength (MPa)**, is the answer the model will learn to predict from the others.

In [ ]:
#@title ▶ Step 1a · Look at the table { display-mode: "form" }
rows = 10 #@param [5, 10, 20] {type:"raw"}
lab.show_table(rows)


In [ ]:
#@title ▶ Step 1b · Guess it yourself { display-mode: "form" }
#@markdown Five mixes without their answer: which grade of compressive strength does each one reach? Pick, click the button, and see. Step 2a shows what the model makes of the same mixes.
lab.guess()


## Part 2 · Two questions, one table

The same table can answer **which class?** (a category: *classification*, the question you just answered yourself) or **how much?** (a number: *regression*). Both models train on 80 % of the rows and are scored on the 20 % they never saw.

In [ ]:
#@title ▶ Step 2a · Classification: predict the class { display-mode: "form" }
#@markdown *grades* puts each mix in one of 3 bands of compressive strength, the game of Step 1b; *pass / fail* asks whether it reaches the *threshold* you set.
model = "decision trees (gradient boosting)" #@param ["a straight line (linear regression)", "decision trees (gradient boosting)"]
task = "grades" #@param ["grades", "pass / fail against a specification"]
threshold = 30 #@param {type:"slider", min:15, max:60, step:5}
lab.classification(model, task, threshold)


In [ ]:
#@title ▶ Step 2b · Regression: predict the number { display-mode: "form" }
model = "decision trees (gradient boosting)" #@param ["a straight line (linear regression)", "decision trees (gradient boosting)"]
lab.regression(model)


> ### 📝 Report question 1
> From Step 2a: how many mixes the model puts in the right grade against your own score in Step 1b, and at your pass / fail threshold how many false passes and false fails there are. From Step 2b: the average miss of the straight line and of the trees, in MPa, and what the worst misses have in common. What is the difference between predicting 'pass' and predicting 33 MPa, and which of the two mistakes costs more on a real project?

## Part 3 · What the model learned

A model that scores well may still have learned the wrong thing. Two checks: which columns it leans on, and how its prediction moves when you change one input at a time.

In [ ]:
#@title ▶ Step 3a · Which columns matter { display-mode: "form" }
lab.importance()


In [ ]:
#@title ▶ Step 3b · What if… { display-mode: "form" }
#@markdown Move a slider; the prediction updates. Everything not on a slider stays as it is in the chosen row.
start_from = "a typical row" #@param ["a typical row", "row 12", "row 100", "row 500"]
lab.whatif(start_from)


> ### 📝 Report question 2
> From Step 3: the three columns that matter most. Does the model agree with what you know about concrete (more water, longer curing, more cement)? Push one slider to the edge of its range: where does the prediction stop making sense, and why can a model not know that?

## Part 4 · The same job, by a chat model

**hokie.ai** (https://hokie.ai.vt.edu/, Virginia Tech's free access to GPT models, sign in with your VT account) gets the same training mixes the models above learned from, and 30 of the held-out mixes without their compressive strength. You paste its reply back into the notebook, which scores it against what really happened, next to the notebook's own models. First the chat on its own, then the chat told to use its **data-analysis tool** (it writes and runs code on the files). The 30 mixes are the same for everyone, so you can compare with your neighbours.

In [ ]:
#@title ▶ Step 4a · Ask the chat { display-mode: "form" }
#@markdown Run the same prompt in **two** new chats and score both replies: the table then compares them. If attaching files does not work, choose *paste the data into the prompt* and run the cell again.
give = "attach the files" #@param ["attach the files", "paste the data into the prompt"]
lab.chat_table(give)


In [ ]:
#@title ▶ Step 4b · Ask the chat to use its analysis tool { display-mode: "form" }
#@markdown Pick the model the chat should train, run the cell, and follow the steps. If the reply shows no code or analysis panel, ask it again to *use your data-analysis tool*. Then try a second model: every reply you score stays in the table.
model = "gradient-boosted trees" #@param ["gradient-boosted trees", "a random forest", "a straight line", "a small neural network"]
lab.chat_table_tool(model)


> ### 📝 Report question 3
> From Step 4a: the chat's average miss and right grades next to the trees', and how many numbers changed between your two new chats. Ask the chat how it made those predictions: what does it say it did? From Step 4b: the model you asked for, its average miss, and why it lands where it does against the notebook's trees and straight line (Step 2b). Compare the three columns the chat said mattered most with Step 3a. When would you trust a chat's numbers on a real project, and what would you check first?

## Part 5 · Your own table

A small app, opened from a link: upload any CSV, pick the column to predict, and it fits decision trees and scores them on held-out rows.

In [ ]:
#@title ▶ Step 5 · Your own table { display-mode: "form" }
#@markdown Open the printed link in a new tab.
lab.upload_app()


> ### 📝 Report question 4
> Upload one table of your own (a cost table, a bid tabulation, anything with a numeric column and 30+ rows) and report what the app found: the score, the columns that mattered, and whether you believe it.

## Wrap-up

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
lab.report_summary()


### Credits
- Table: Concrete Compressive Strength, I-Cheng Yeh (1998), UCI Machine Learning Repository, CC BY 4.0, https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength.
- Chat model: the GPT models behind hokie.ai (Virginia Tech).
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp6_tabular_timeseries`).